![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

## Resumen

Se cargó el archivo `bank_marketing.csv`, se revisó su estructura inicial y se dividió en tres DataFrames: `client`, `campaign` y `economics`.

Durante el proceso se limpiaron variables de texto, se trataron valores desconocidos, se convirtieron columnas binarias a booleanas y se creó la fecha `last_contact_date` en formato datetime.

In [39]:
# Importa pandas para trabajar con DataFrames
import pandas as pd

# Importa numpy para manejar valores nulos como np.nan y operaciones condicionales
import numpy as np

# Lee el archivo CSV original y lo guarda en el DataFrame df
df = pd.read_csv("bank_marketing.csv")

In [40]:
# Muestra las primeras filas del dataset para revisar su estructura general
df.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no


In [41]:
# Muestra el número de filas y columnas del DataFrame
df.shape

(41188, 16)

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_id                   41188 non-null  int64  
 1   age                         41188 non-null  int64  
 2   job                         41188 non-null  object 
 3   marital                     41188 non-null  object 
 4   education                   41188 non-null  object 
 5   credit_default              41188 non-null  object 
 6   mortgage                    41188 non-null  object 
 7   month                       41188 non-null  object 
 8   day                         41188 non-null  int64  
 9   contact_duration            41188 non-null  int64  
 10  number_contacts             41188 non-null  int64  
 11  previous_campaign_contacts  41188 non-null  int64  
 12  previous_outcome            41188 non-null  object 
 13  cons_price_idx              411

In [43]:
# Cuenta la cantidad de valores nulos por columna
df.isna().sum()

client_id                     0
age                           0
job                           0
marital                       0
education                     0
credit_default                0
mortgage                      0
month                         0
day                           0
contact_duration              0
number_contacts               0
previous_campaign_contacts    0
previous_outcome              0
cons_price_idx                0
euribor_three_months          0
campaign_outcome              0
dtype: int64

## Parte 1: Revisión de valores categóricos

Se revisan las variables categóricas que requieren limpieza o conversión de formato.  
Esto permite identificar valores como `yes`, `no`, `unknown`, `success`, `failure` o `nonexistent`, y definir cómo deben transformarse según los requerimientos del ejercicio.

In [44]:
# Lista de columnas categóricas que necesitan revisión antes de ser transformadas
cols_revision = ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]

# Recorre cada columna de la lista
for col in cols_revision:
    
    # Imprime el nombre de la columna que se está revisando
    print(col)
    
    # Muestra la frecuencia de cada valor dentro de la columna
    print(df[col].value_counts())
    
    # Imprime una línea separadora para ordenar la salida
    print("-------------")

credit_default
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
-------------
mortgage
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64
-------------
previous_outcome
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
-------------
campaign_outcome
no     36548
yes     4640
Name: campaign_outcome, dtype: int64
-------------


## Parte 2: Creación del DataFrame client

Se crea el DataFrame `client` con la información demográfica y financiera del cliente.  
En esta etapa se seleccionan las columnas requeridas, se limpian valores de texto y se convierten las variables binarias al tipo booleano según las reglas indicadas.

In [45]:
# Crea el DataFrame client seleccionando solo las columnas requeridas .copy() evita modificar accidentalmente el DataFrame original df
client = df[[
    "client_id",
    "age",
    "job",
    "marital",
    "education",
    "credit_default",
    "mortgage"
]].copy()

# Reemplaza puntos por guiones bajos en la columna job
# regex=False indica que el punto se interpreta como texto literal
client["job"] = client["job"].str.replace(".", "_", regex=False)

# Reemplaza puntos por guiones bajos en la columna education
client["education"] = client["education"].str.replace(".", "_", regex=False)

# Convierte el valor "unknown" en NaN para representar dato faltante
client["education"] = client["education"].replace("unknown", np.nan)

# Convierte credit_default a binario:
# 1 si el valor es "yes", 0 en cualquier otro caso
client["credit_default"] = np.where(client["credit_default"] == "yes", 1, 0)

# Convierte mortgage a binario:
# 1 si el valor es "yes", 0 en cualquier otro caso
client["mortgage"] = np.where(client["mortgage"] == "yes", 1, 0)

# Cambia credit_default a tipo booleano
client["credit_default"] = client["credit_default"].astype(bool)

# Cambia mortgage a tipo booleano
client["mortgage"] = client["mortgage"].astype(bool)

# Revisa las primeras filas del DataFrame client
client.head()

,client_id,age,job,marital,education,credit_default,mortgage
0,0,56,housemaid,married,basic_4y,False,False
1,1,57,services,married,high_school,False,False
2,2,37,services,married,high_school,False,True
3,3,40,admin_,married,basic_6y,False,False
4,4,56,services,married,high_school,False,False


In [46]:
# Verifica los tipos de datos después de la limpieza
client.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   client_id       41188 non-null  int64 
 1   age             41188 non-null  int64 
 2   job             41188 non-null  object
 3   marital         41188 non-null  object
 4   education       39457 non-null  object
 5   credit_default  41188 non-null  bool  
 6   mortgage        41188 non-null  bool  
dtypes: bool(2), int64(2), object(3)
memory usage: 1.6+ MB


## Parte 3: Creación del DataFrame campaign

Se crea el DataFrame `campaign` con la información asociada a los contactos de campaña.  
Se convierten las variables de resultado a booleanas y se construye la columna `last_contact_date` combinando día, mes y año fijo 2022.

In [47]:
# Crea el DataFrame campaign con las columnas requeridas
campaign = df[[
    "client_id",
    "number_contacts",
    "contact_duration",
    "previous_campaign_contacts",
    "previous_outcome",
    "campaign_outcome",
    "day",
    "month"
]].copy()

# Convierte previous_outcome a binario:
# 1 si el resultado anterior fue "success", 0 en cualquier otro caso
campaign["previous_outcome"] = np.where(campaign["previous_outcome"] == "success", 1, 0)

# Convierte campaign_outcome a binario:
# 1 si el resultado de la campaña actual fue "yes", 0 en cualquier otro caso
campaign["campaign_outcome"] = np.where(campaign["campaign_outcome"] == "yes", 1, 0)

# Cambia previous_outcome a tipo booleano
campaign["previous_outcome"] = campaign["previous_outcome"].astype(bool)

# Cambia campaign_outcome a tipo booleano
campaign["campaign_outcome"] = campaign["campaign_outcome"].astype(bool)

# Crea una columna year con valor fijo 2022, según el requerimiento
campaign["year"] = 2022

# Construye la fecha de último contacto combinando year, month y day
# El formato esperado es YYYY-MMM-DD, por ejemplo: 2022-may-15
campaign["last_contact_date"] = pd.to_datetime(
    campaign["year"].astype(str) + "-" +
    campaign["month"].astype(str) + "-" +
    campaign["day"].astype(str),
    format="%Y-%b-%d"
)

# Elimina las columnas auxiliares que ya fueron usadas para crear last_contact_date
campaign = campaign.drop(columns=["day", "month", "year"])

# Revisa las primeras filas del DataFrame campaign
campaign.head()

,client_id,number_contacts,contact_duration,previous_campaign_contacts,previous_outcome,campaign_outcome,last_contact_date
0,0,1,261,0,False,False,2022-05-13
1,1,1,149,0,False,False,2022-05-19
2,2,1,226,0,False,False,2022-05-23
3,3,1,151,0,False,False,2022-05-27
4,4,1,307,0,False,False,2022-05-03


In [48]:
# Verifica los tipos de datos después de la limpieza
campaign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   client_id                   41188 non-null  int64         
 1   number_contacts             41188 non-null  int64         
 2   contact_duration            41188 non-null  int64         
 3   previous_campaign_contacts  41188 non-null  int64         
 4   previous_outcome            41188 non-null  bool          
 5   campaign_outcome            41188 non-null  bool          
 6   last_contact_date           41188 non-null  datetime64[ns]
dtypes: bool(2), datetime64[ns](1), int64(4)
memory usage: 1.6 MB


## Parte 4: Creación del DataFrame economics

Se crea el DataFrame `economics` con las variables económicas asociadas a cada cliente.  
Estas columnas no requieren transformaciones adicionales según los requisitos, pero se separan en una fuente independiente para mantener una estructura ordenada.

In [49]:
# Crea el DataFrame economics seleccionando las columnas económicas requeridas
economics = df[[
    "client_id",
    "cons_price_idx",
    "euribor_three_months"
]].copy()

# Revisa las primeras filas del DataFrame economics
economics.head()

,client_id,cons_price_idx,euribor_three_months
0,0,93.994,4.857
1,1,93.994,4.857
2,2,93.994,4.857
3,3,93.994,4.857
4,4,93.994,4.857


In [50]:
# Verifica los tipos de datos de economics
economics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             41188 non-null  int64  
 1   cons_price_idx        41188 non-null  float64
 2   euribor_three_months  41188 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 965.5 KB


## Parte 5: Validación y exportación de archivos

Finalmente, se valida que los tres DataFrames existan y tengan la estructura esperada.  
Luego se exportan como archivos CSV sin índice, cumpliendo con los nombres solicitados: `client.csv`, `campaign.csv` y `economics.csv`.

In [51]:
# Revisa la cantidad de filas y columnas de client
client.shape

(41188, 7)

In [52]:
# Revisa la cantidad de filas y columnas de campaign
campaign.shape


(41188, 7)

In [53]:
# Revisa la cantidad de filas y columnas de economics
economics.shape

(41188, 3)

In [54]:
# Exporta el DataFrame client como archivo CSV sin índice
client.to_csv("client.csv", index=False)

# Exporta el DataFrame campaign como archivo CSV sin índice
campaign.to_csv("campaign.csv", index=False)

# Exporta el DataFrame economics como archivo CSV sin índice
economics.to_csv("economics.csv", index=False)

In [55]:
# Lee nuevamente los archivos exportados para comprobar que fueron guardados correctamente
client_check = pd.read_csv("client.csv")
campaign_check = pd.read_csv("campaign.csv")
economics_check = pd.read_csv("economics.csv")

# Muestra las dimensiones de los archivos exportados
print("client.csv:", client_check.shape)
print("campaign.csv:", campaign_check.shape)
print("economics.csv:", economics_check.shape)

client.csv: (41188, 7)
campaign.csv: (41188, 7)
economics.csv: (41188, 3)


## Conclusión

El dataset original quedó separado en tres archivos limpios y organizados: `client.csv`, `campaign.csv` y `economics.csv`.

Esta estructura facilita el análisis posterior de clientes, campañas e indicadores económicos, manteniendo datos consistentes, trazables y listos para uso analítico.